# 06 Control: PD And Impedance Intuition

Planning 给期望轨迹，control 处理真实动态响应。

![PD and impedance control visual map](assets/figures/06_control_pd_impedance.png)

## Learning Objectives

- Interpret proportional gain as position-error correction.
- Interpret derivative gain as damping.
- Connect control tuning to overshoot, settling, and force limits.

## Checkpoint

- Run the PD simulation and read the final position, velocity, and max control.
- Explain what changes when damping is reduced.
- Name one reason a stable simulation may still be unsafe on hardware.

## Practice Task

Try three `kd` values while keeping `kp` fixed. Summarize which response you would trust most for a real manipulator and why.

## Concept Map

![Colab concept image](assets/colab/06_control_pd_impedance_concept.png)

**Concept image.** PD control can be read as a virtual spring and damper driving position error toward zero.

In [ ]:
from pathlib import Path
import sys

COLAB_REPO_URL = "https://github.com/Hollis36/robotic-manipulation-learning.git"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    in_colab = "google.colab" in sys.modules
    if in_colab:
        import subprocess

        PROJECT_ROOT = Path("/content/robotic-manipulation-learning")
        if not PROJECT_ROOT.exists():
            # Equivalent command: git clone --depth 1 <repo> <target>
            subprocess.run(["git", "clone", "--depth", "1", COLAB_REPO_URL, str(PROJECT_ROOT)], check=True)
    else:
        PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

PROJECT_ROOT


In [ ]:
from rml.control import simulate_pd_mass

trace = simulate_pd_mass(x0=0.0, v0=0.0, target=1.0, kp=25.0, kd=10.0, dt=0.01, steps=500)
trace.positions[-1], trace.velocities[-1], max(abs(trace.controls))

## Result Figure

Plot the simulated position response against the target to inspect overshoot and settling.

The figure below is generated from the values computed in this notebook. Treat it as evidence from the code, not as a decorative illustration.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.figsize": (7, 4.2),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})
target = 1.0
fig, ax = plt.subplots()
ax.plot(trace.times, trace.positions, lw=3, label='position')
ax.axhline(target, color='#E45756', linestyle='--', label='target')
ax.set_xlabel('time (s)')
ax.set_ylabel('position')
ax.legend(frameon=False)
plt.show()

## Parameter Experiment

The next cell is marked with `COLAB_PARAMETER_EXPERIMENT` so it is easy to find in Colab. Change derivative gain values and compare the maximum overshoot for each response.

In [ ]:
# COLAB_PARAMETER_EXPERIMENT
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.figsize": (7, 4.2),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})
import numpy as np

kd_values = [2.0, 6.0, 10.0, 18.0]
fig, ax = plt.subplots()
for kd in kd_values:
    trial = simulate_pd_mass(x0=0.0, v0=0.0, target=1.0, kp=25.0, kd=kd, dt=0.01, steps=500)
    overshoot = max(0.0, float(np.max(trial.positions) - 1.0))
    ax.plot(trial.times, trial.positions, label=f'kd={kd}')
    print('kd=', kd, 'max_overshoot=', round(overshoot, 4))
ax.axhline(1.0, color='#E45756', linestyle='--', label='target')
ax.set_xlabel('time (s)')
ax.set_ylabel('position')
ax.legend(frameon=False)
plt.show()

## Reflection Prompt

如果 `kd` 太小或太大，真实机械臂上分别可能出现什么问题？用 overshoot、settling 和控制力限制回答。

Exercise: 降低 `kd`，观察系统是否更容易振荡。